### Pre-modeling

In [82]:
import pandas as pd
import numpy as np
import time
import mlflow
import mlflow.lightgbm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
df = pd.read_csv('../data/processed/Telco-Customer-Churn-Processed.csv')

In [3]:
X = df.drop(columns=['Churn'])
y = df['Churn']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=86, stratify=y
)

---

We have to choose which metric to use.

In our business problem:
- if model makes type 1 error -> company will trigger its standard retention protocols (discounts, sending gifts etc.) basically unnecessary costs

- if model makes type 2 error -> customer will leave, which will decrease revenue of the company, it also leads to damage to reputation and to spend more money on marketing

In [5]:
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

We see that dataset is imbalanced, so accuracy can't be used there.

Precision is also not the best option because cost of false positive is not that bad.

Cost of false negative on the other side is too high, so the best option is `recall` or `f1`, but i would stick up to recall, since FN is more dangerous and `threshold` in our case would be `0.3`

---

### Model selection (RF vs LGBM vs XGB)

In [6]:
# Robust to class imbalance 
ratio = (y_train == 0).sum() / (y_train == 1).sum()
THRESHOLD = 0.3

#### Random Forest 

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight='balanced',
    random_state=75,
    n_jobs=-1
)

rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.50      0.66      1035
           1       0.41      0.95      0.57       374

    accuracy                           0.62      1409
   macro avg       0.69      0.73      0.62      1409
weighted avg       0.82      0.62      0.64      1409



Based on classification report we see, that model found 95% of churn customers

But on the other side it predicts only 41% of them correctly -> Model's making a lot of type 1 error    

---

#### LGBM

In [8]:
from lightgbm import LGBMClassifier

In [9]:
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight='balanced',
    random_state=75,
    n_jobs=-1
)

start = time.perf_counter()

lgbm.fit(X_train, y_train)

proba = lgbm.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

end = time.perf_counter()

print(f"Training took {end - start:.3f} seconds")

print(classification_report(y_test, y_pred))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002406 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

Model caught 83% of actual churn customers (13% worse than baseline) -> which is overall not that bad, but model makes a lot of false positive mistakes (only 50% of predicted churns are actually churns) 

---

#### XGBoost

In [10]:
from xgboost import XGBClassifier

In [11]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    class_weight='balanced',
    random_state=75,
    n_jobs=-1
)

start = time.perf_counter()

xgb.fit(X_train, y_train)

proba = xgb.predict_proba(X_test)[:, 1]
y_pred = (proba >= THRESHOLD).astype(int)

end = time.perf_counter()

print(f"Training took {end - start:.3f} seconds")
print(classification_report(y_test, y_pred))

d:\reps\churn-customer\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:16:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training took 1.763 seconds
              precision    recall  f1-score   support

           0       0.87      0.82      0.85      1035
           1       0.58      0.67      0.62       374

    accuracy                           0.78      1409
   macro avg       0.73      0.75      0.73      1409
weighted avg       0.79      0.78      0.79      1409



XGBoost has slightly better accuracy, precision and it's 38% faster than LGBM

But on the recall metric it managed to find only 67% of all churns (16% worse)

---

I'm gonna choose LGBM since it managed to find more churn customers which for company is more significant since FN cost is very high, but not FP

Yes, i know that i'm gonna sacrifice speed and overall accuracy, but what this'll cost to company? Additional 10$ discount offer instead of customer leaving

In real world i'd decide what to sacrifice with company but since it's pet project i'm choosing `LGBM`

---

### Threshold testing

In [12]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = lgbm.predict_proba(X_test)[:, 1]

for t in [0.2, 0.25, 0.3, 0.35, 0.4]:
    preds = (proba >= t).astype(int)
    prec = precision_score(y_test, preds, pos_label=1)
    rec = recall_score(y_test, preds, pos_label=1)
    f1 = f1_score(y_test, preds, pos_label=1)
    print(f"Threshold: {t} has precision: {prec:.3f}, recall: {rec:.3f}, f1: {f1:.3f}")

Threshold: 0.2 has precision: 0.449, recall: 0.885, f1: 0.595
Threshold: 0.25 has precision: 0.473, recall: 0.853, f1: 0.609
Threshold: 0.3 has precision: 0.498, recall: 0.826, f1: 0.622
Threshold: 0.35 has precision: 0.516, recall: 0.799, f1: 0.627
Threshold: 0.4 has precision: 0.538, recall: 0.783, f1: 0.638


Best precision-recall tradeoff found on 0.25 threshold, so i'll use it

In [13]:
THRESHOLD = 0.25

---

### Hyperparameters tuning 

In [69]:
import optuna
from lightgbm import LGBMClassifier, early_stopping
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score

In [95]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

In [96]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 20, log=True),

        "subsample_freq": 1,
        "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
        "random_state": 87,
        "n_jobs": -1,
        "objective": "binary",
        "verbosity": -1
    }

    model = LGBMClassifier(**params)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[early_stopping(50, verbose=False)]
    )

    proba = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y_val, proba)

In [97]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

[I 2026-07-21 13:18:06,075] A new study created in memory with name: no-name-cb7217d7-9a50-4aa4-aa20-dd1cc9e8349c
[I 2026-07-21 13:18:08,107] Trial 0 finished with value: 0.8692622752169066 and parameters: {'n_estimators': 1195, 'learning_rate': 0.03541451112780508, 'max_depth': 5, 'num_leaves': 49, 'subsample': 0.9223901995842823, 'colsample_bytree': 0.5742515580917689, 'min_child_samples': 95, 'min_split_gain': 0.7728319823882915, 'min_child_weight': 0.009122767189166854, 'reg_lambda': 0.867632799946353, 'reg_alpha': 3.7695983643827002}. Best is trial 0 with value: 0.8692622752169066.
[I 2026-07-21 13:18:08,932] Trial 1 finished with value: 0.8667518943983972 and parameters: {'n_estimators': 1160, 'learning_rate': 0.07108366744047769, 'max_depth': 5, 'num_leaves': 109, 'subsample': 0.9306438592718231, 'colsample_bytree': 0.9667192516300094, 'min_child_samples': 12, 'min_split_gain': 0.16977945038098707, 'min_child_weight': 0.710024815322804, 'reg_lambda': 0.02308544708001995, 'reg_al

---

### Best model & MLFlow

In [98]:
best_params = study.best_params

print(f"Best params: {study.best_params}")
print(f"Best score: {study.best_value}")

Best params: {'n_estimators': 417, 'learning_rate': 0.057311568902010564, 'max_depth': 3, 'num_leaves': 68, 'subsample': 0.7542919655030944, 'colsample_bytree': 0.554260186799192, 'min_child_samples': 47, 'min_split_gain': 0.3861768572644915, 'min_child_weight': 0.0010504318478460233, 'reg_lambda': 0.09124356494212908, 'reg_alpha': 0.36276736858344677}
Best score: 0.8720291470764061


In [99]:
best_model = LGBMClassifier(**best_params)
best_model.fit(X_train, y_train)

,num_leaves,68
,max_depth,3
,learning_rate,0.057311568902010564
,n_estimators,417
,min_split_gain,0.3861768572644915
,min_child_weight,0.0010504318478460233
,min_child_samples,47
,subsample,0.7542919655030944
,colsample_bytree,0.554260186799192
,reg_alpha,0.36276736858344677
,reg_lambda,0.09124356494212908


In [100]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score
)

train_prob = best_model.predict_proba(X_train)[:, 1]
train_pred = (train_prob >= THRESHOLD).astype(int)

test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= THRESHOLD).astype(int)


metrics = {
    "train_accuracy": accuracy_score(y_train, train_pred),
    "test_accuracy": accuracy_score(y_test, test_pred),

    "train_f1": f1_score(y_train, train_pred),
    "test_f1": f1_score(y_test, test_pred),

    "train_roc_auc": roc_auc_score(y_train, train_prob),
    "test_roc_auc": roc_auc_score(y_test, test_prob),

    "train_recall": recall_score(y_train, train_pred),
    "test_recall": recall_score(y_test, test_pred)
}

gap_metrics = {
    "accuracy_gap": metrics["train_accuracy"] - metrics["test_accuracy"],
    "f1_gap": metrics["train_f1"] - metrics["test_f1"],
    "roc_auc_gap": metrics["train_roc_auc"] - metrics["test_roc_auc"],
    "recall_gap": metrics["train_recall"] - metrics["test_recall"]
}

In [101]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Churn LGBM model comparison")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1784628754694, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1784628754694, lifecycle_stage='active', name='Churn LGBM model comparison', tags={}, trace_location=None, workspace='default'>

In [106]:
with mlflow.start_run(run_name="LGBM default"):
    mlflow.lightgbm.log_model(best_model, name="no-cv")
    mlflow.log_params(best_params)
    mlflow.log_param("threshold", THRESHOLD)
    mlflow.log_metrics(metrics)
    mlflow.log_metrics(gap_metrics)

2026/07/21 15:02:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run LGBM default at: http://127.0.0.1:5000/#/experiments/2/runs/9d3ad6066b364671a98222fd853db978
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### Overfitting testing

In [104]:
scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="roc_auc"
)

print(scores.mean(), scores.std())

0.844240766688839 0.006136398332650664


---

I tried different optuna params tuning and here's what i've achieved:

Pure optuna with no cv or early stopping: tuning lasts `30 seconds`:
- roc_auc: train - `0.87`, test - `0.86`, gap - `0.01`
- f1: train - `0.65`, test - `0.65`, gap - `0.00`
- recall: train - `0.84`, test - `0.84`, gap -`0.00`

With cv: almost `6 minutes`:
- roc_auc: train - `0.86`, test - `0.86`, gap - `0.00`
- f1: train - `0.65`, test - `0.65`, gap - `0.00`
- recall: train - `0.85`, test - `0.86`, gap - neg. `0.01`

With early stopping: `3 minute 15 seconds`:
- roc_auc: train - `0.91`, test - `0.86`, gap - `0.05`
- f1: train - `0.7`, test - `0.65`, gap - `0.05`
- recall: train - `0.9`, test - `0.82`, gap - `0.08`

---

### Summary

---

3 tuning strategies were evaluated:

- Optuna without CV
- Optuna with 5-fold Stratified CV
- Optuna with Stratified CV and early stopping

While CV increased computational cost (30 seconds → 6 minutes), it didn't improve final ROC AUC score, which remained approximately `0.86.`

Adding early stopping reduced runtime compared to full CV (3 minutes 15 sec.) but resulted in model overfitting and lower recall on the test set. 

Suprisingly, for this dataset the simpler Optuna tuning approach achieved the best trade-off between runtime and performance, so i'll use this model

---

#### The problems that i faced while doing this project:
- I was using default threshold (0.5) intead of custom one (0.25) which in result lead to wrong metrics results

Because of this most of my predictions has failed and i got both F1 scores 0.000, but it was quickly fixed by switching back to custom 0.25 threshold

- Firstly, when i was testing optuna I didn't register models, but when i decided to move to early stopping version of params tuning i connected project to MLFlow

This issue took additional 20 minutes of my time

---

#### Trade-offs

The most obvious one is precision-recall trade-off, which in our case recall is higher than precision 
- that means model will mark loyal customers as churns pretty often -> company'll spend additional money on "bringing back" customer, but thanks to it model also will capture `84%` of all churn customers

---

#### Bisiness problem solving

Model can help telco company to determine whether customer will leave or not -> will decreases costs of ads, campaings, etc.

Let's imagine a scenario in which there're 1 million of customers, with this new model i calculated that:

- 409,756 subscribers will be flagged by model

- 168,000 of them are actual churns

- 241,756 of them are fake alarms

- about 67,200 customers will be saved

- Gross Revenue Saved - $40,320,000

- Total Campaign Cost - $20,487,800

- Net Annual Savings - $19,832,200

Still model misses 16% of churns and company has to offer discounts for 240k fake alarms customers, but overall savings are higher than costs

---